In [0]:

CATALOG = "science_home"
SCHEMA = "`ml-dielectric`"
SOURCE_TABLE = f"{CATALOG}.`db-inorganics`.mp_gold"

# Table names
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.dielectric_silver"
BP_FEATURIZER_TABLE = f"{CATALOG}.{SCHEMA}.bp_dielectric_featurizer"
MATMINER_TABLE = f"{CATALOG}.{SCHEMA}.compos_matminer_features"
FEATURIZED_MERGED_TABLE = f"{CATALOG}.{SCHEMA}.featurized_merged"
TRAIN_TABLE = f"{CATALOG}.{SCHEMA}.train_data"
TEST_TABLE = f"{CATALOG}.{SCHEMA}.test_data"
PREDICTION_TABLE = f"{CATALOG}.{SCHEMA}.prediction_data"
MODEL_NAME_UC = f"{CATALOG}.ml-dielectric.xgboost_dielectric"
MLFLOW_EXPERIMENT = "/Users/j.krishna@exomatter.ai/dielectric_model_comparison"

# Columns to drop 
NULL_COLS_TO_DROP = [
    'types_of_magnetic_species', 'possible_species', 'elements',
    'weighted_work_function', 'weighted_surface_energy',
    'weighted_surface_energy_ev_per_ang_2', 'surface_energy',
    'surface_anisotropy', 'shape_factor', 'e_ij_max',
    'e_ionic', 'e_electronic',
]
MAGNETIC_COLS_TO_DROP = [
    'num_magnetic_sites', 'num_unique_magnetic_sites',
    'total_magnetization', 'total_magnetization_normalized_formula_units',
    'total_magnetization_normalized_vol',
    'LUMO_character', 'HOMO_element', 'HOMO_character', 'LUMO_element',
]
ID_COLS = ['id', 'formula']
TARGET = 'dielectric_constant'

# Defaults
OUTLIER_THRESHOLD = 30        # keep dielectric_constant <= 30
SELECTED_ACSF = ['acsf_3761'] # top correlated ACSF feature(s)
TEST_SIZE = 0.2
RANDOM_STATE = 42
STRATIFY_BINS = 10

print("All the configurations are loaded.")

Configuration loaded.


In [0]:
#  DATA INGESTION


def load_source_data(table_name=SOURCE_TABLE):
    """Read the raw Materials Project gold table from Unity Catalog."""
    df = spark.table(table_name)
    print(f"Loaded {table_name}: {df.count()} rows, {len(df.columns)} cols")
    return df


def drop_non_feature_columns(df):
    """
    Drop all string columns (except 'id'), boolean columns,
    and specific list/array columns that are not useful features.
    """
    string_cols = [name for name, dtype in df.dtypes if dtype == 'string' and name != 'id']
    bool_cols = [name for name, dtype in df.dtypes if dtype == 'boolean']
    extra_cols = ['types_of_magnetic_species', 'possible_species', 'elements']
    cols_to_drop = string_cols + bool_cols + extra_cols
    df_out = df.drop(*cols_to_drop)
    print(f"Dropped {len(cols_to_drop)} non-feature columns → {len(df_out.columns)} remain")
    return df_out


def drop_high_null_columns(df, cols_to_drop=None):
    """
    Drop columns that are predominantly null.
    Uses NULL_COLS_TO_DROP by default.
    """
    cols_to_drop = cols_to_drop or NULL_COLS_TO_DROP
    existing = [c for c in cols_to_drop if c in df.columns]
    df_out = df.drop(*existing)
    print(f"Dropped {len(existing)} high-null columns → {len(df_out.columns)} remain")
    return df_out


def save_to_delta(df, table_name, mode="overwrite"):
    """Write a Spark DataFrame to a Unity Catalog Delta table."""
    df.write.mode(mode).format("delta").saveAsTable(table_name)
    print(f"Saved to {table_name} ({mode})")


def run_data_ingestion():
    """
    Full Step 1 pipeline:
    Load mp_gold → drop non-feature columns → drop high-null columns → save silver.
    """
    df = load_source_data()
    df = drop_non_feature_columns(df)
    df = drop_high_null_columns(df)
    save_to_delta(df, SILVER_TABLE)
    return df



Step 1 functions defined.


In [0]:

#  FEATURE MERGING

def load_feature_tables():
    """
    Load the three feature sources from Unity Catalog:
    silver data, BP (ACSF) featurizer, and Matminer composition features.
    """
    df_silver = spark.table(SILVER_TABLE)
    df_bp = spark.table(BP_FEATURIZER_TABLE)
    df_magpie = spark.table(MATMINER_TABLE)
    for name, df in [("Silver", df_silver), ("BP featurizer", df_bp), ("Matminer", df_magpie)]:
        print(f"{name}: {df.count()} rows, {len(df.columns)} cols")
    return df_silver, df_bp, df_magpie


def merge_features(df_silver, df_bp, df_magpie):
    """
    Inner-join all three DataFrames on 'id'.
    Drops duplicate 'formula' from Matminer before joining.
    """
    df_merged = df_bp.join(df_silver, on="id", how="inner")
    df_final = df_merged.join(df_magpie.drop("formula"), on="id", how="inner")
    print(f"Merged: {df_final.count()} rows, {len(df_final.columns)} cols")
    return df_final


def run_feature_merging():
    """
    Full Step 2 pipeline:
    Load three feature tables → merge → save featurized_merged.
    """
    df_silver, df_bp, df_magpie = load_feature_tables()
    df_final = merge_features(df_silver, df_bp, df_magpie)
    save_to_delta(df_final, FEATURIZED_MERGED_TABLE)
    return df_final



Step 2 functions defined.


In [0]:

# FEATURE SELECTION & DATA PREPARATION

import numpy as np
import pandas as pd
from pyspark.sql.functions import (
    col, log as spark_log,
    min as spark_min, max as spark_max,
    mean as spark_mean, stddev, skewness,
    count, when,
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


def drop_magnetic_columns(df, cols=None):
    """Drop magnetic and HOMO/LUMO columns not useful for dielectric prediction."""
    cols = cols or MAGNETIC_COLS_TO_DROP
    existing = [c for c in cols if c in df.columns]
    df_out = df.drop(*existing)
    print(f"Dropped {len(existing)} magnetic/orbital columns → {len(df_out.columns)} remain")
    return df_out


def select_acsf_features(df, selected_acsf=None):
    """
    Keep only the selected ACSF features (by default the single
    most-correlated one: acsf_3761) plus all non-ACSF columns.
    """
    selected_acsf = selected_acsf or SELECTED_ACSF
    non_acsf_cols = [c for c in df.columns if not c.startswith('acsf_')]
    keep_cols = non_acsf_cols + selected_acsf
    df_out = df.select(keep_cols)
    print(f"Kept {len(selected_acsf)} ACSF + {len(non_acsf_cols)} other = {len(keep_cols)} total cols")
    return df_out


def filter_non_null_target(df, target_col=TARGET):
    """Keep only rows where the target column is not null."""
    before = df.count()
    df_out = df.dropna(subset=[target_col])
    after = df_out.count()
    print(f"Non-null target filter: {before} → {after} rows")
    return df_out


def remove_outliers(df, target_col=TARGET, threshold=OUTLIER_THRESHOLD):
    """Remove rows where target exceeds the given threshold."""
    before = df.count()
    df_out = df.filter(col(target_col) <= threshold)
    after = df_out.count()
    print(f"Outlier removal ({target_col} <= {threshold}): {before} → {after} rows")
    return df_out


def log_transform_target(df, target_col=TARGET):
    """Apply natural-log transform to the target column (in-place replacement)."""
    df_out = df.withColumn(target_col, spark_log(col(target_col)))
    stats = df_out.select(
        spark_min(target_col).alias('min'),
        spark_max(target_col).alias('max'),
        spark_mean(target_col).alias('mean'),
        stddev(target_col).alias('std'),
        skewness(target_col).alias('skew'),
    ).collect()[0]
    print(f"Log-transform applied. min={stats['min']:.4f}, max={stats['max']:.4f}, "
          f"mean={stats['mean']:.4f}, skew={stats['skew']:.4f}")
    return df_out


def impute_nulls_with_zero(df, exclude_cols=None):
    """
    Fill null values with 0 for all numeric feature columns.
    Excludes id, formula, and target by default.
    """
    exclude_cols = exclude_cols or ID_COLS + [TARGET]
    numeric_cols = [
        c for c in df.columns
        if c not in exclude_cols
        and df.schema[c].dataType.typeName() in ('double', 'float', 'integer', 'long')
    ]
    df_out = df.fillna(0, subset=numeric_cols)
    print(f"Imputed {len(numeric_cols)} numeric feature columns with 0")
    return df_out, numeric_cols


def scale_features(df_pd, exclude_cols=None):
    """
    StandardScaler on numeric feature columns (pandas DataFrame).
    Returns (scaled_df, scaler, feature_col_names).
    """
    exclude_cols = exclude_cols or ID_COLS + [TARGET]
    feature_cols = [
        c for c in df_pd.select_dtypes(include=['number']).columns
        if c not in exclude_cols
    ]
    scaler = StandardScaler()
    df_pd[feature_cols] = scaler.fit_transform(df_pd[feature_cols])
    print(f"Scaled {len(feature_cols)} feature columns")
    return df_pd, scaler, feature_cols


def stratified_train_test_split(df_pd, target_col=TARGET, test_size=TEST_SIZE,
                                n_bins=STRATIFY_BINS, random_state=RANDOM_STATE):
    """
    Stratified split using quantile bins on the (log-transformed) target.
    Returns (train_df, test_df) as pandas DataFrames.
    """
    df_pd['_target_bin'] = pd.qcut(df_pd[target_col], q=n_bins, labels=False, duplicates='drop')
    train_df, test_df = train_test_split(
        df_pd, test_size=test_size, random_state=random_state, stratify=df_pd['_target_bin']
    )
    train_df = train_df.drop(columns=['_target_bin'])
    test_df = test_df.drop(columns=['_target_bin'])
    print(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows")
    return train_df, test_df


def prepare_prediction_data(df_selected, scaler, feature_cols, numeric_cols):
    """
    Prepare rows with null target (unseen materials) for prediction.
    Applies the same imputation and scaling as training data.
    Returns a pandas DataFrame.
    """
    df_pred = df_selected.filter(col(TARGET).isNull())
    df_pred_pd = df_pred.fillna(0, subset=numeric_cols).toPandas()
    df_pred_pd[feature_cols] = scaler.transform(df_pred_pd[feature_cols])
    print(f"Prediction data prepared: {len(df_pred_pd)} rows")
    return df_pred_pd


def run_feature_selection_and_split():
    """
    Full Step 3 pipeline:
    Load featurized_merged → drop magnetic cols → select ACSF → filter target nulls
    → remove outliers → log-transform → impute → scale → stratified split
    → save train/test/prediction data to UC.
    Returns (train_df, test_df, pred_df, scaler, feature_cols).
    """
    df = spark.table(FEATURIZED_MERGED_TABLE)
    df = drop_magnetic_columns(df)
    df_selected = select_acsf_features(df)
    df_target = filter_non_null_target(df_selected)
    df_clean = remove_outliers(df_target)
    df_clean = log_transform_target(df_clean)
    df_imputed, numeric_cols = impute_nulls_with_zero(df_clean)

    df_pd = df_imputed.toPandas()
    df_pd, scaler, feature_cols = scale_features(df_pd)
    train_df, test_df = stratified_train_test_split(df_pd)

    # Save train/test
    spark.createDataFrame(train_df).write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(TRAIN_TABLE)
    spark.createDataFrame(test_df).write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(TEST_TABLE)
    print(f"Saved train → {TRAIN_TABLE}, test → {TEST_TABLE}")

    # Prediction data (null target rows)
    pred_df = prepare_prediction_data(df_selected, scaler, feature_cols, numeric_cols)
    spark.createDataFrame(pred_df).write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable(PREDICTION_TABLE)
    print(f"Saved prediction data → {PREDICTION_TABLE}")

    return train_df, test_df, pred_df, scaler, feature_cols



Step 3 functions defined.


In [0]:

# MODEL BUILDING & TUNING

import mlflow
import mlflow.sklearn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV


def load_train_test_pandas():
    """Load train and test DataFrames from Unity Catalog as pandas."""
    train_df = spark.table(TRAIN_TABLE).toPandas()
    test_df = spark.table(TEST_TABLE).toPandas()
    print(f"Train: {train_df.shape} | Test: {test_df.shape}")
    return train_df, test_df


def prepare_xy(train_df, test_df, target_col=TARGET, drop_cols=None):
    """
    Separate features (X) and target (y) from train/test DataFrames.
    Drops non-numeric and identifier columns.
    """
    drop_cols = drop_cols or ID_COLS
    train_df = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns], errors='ignore')
    test_df = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns], errors='ignore')

    X_train = train_df.drop(columns=[target_col]).select_dtypes(include=[np.number])
    y_train = train_df[target_col].dropna()
    X_train = X_train.loc[y_train.index]

    X_test = test_df.drop(columns=[target_col]).select_dtypes(include=[np.number])
    y_test = test_df[target_col].dropna()
    X_test = X_test.loc[y_test.index]
    X_test = X_test[X_train.columns]  # align columns

    print(f"Features: {X_train.shape[1]} | Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
    return X_train, y_train, X_test, y_test


def compute_metrics(y_true, y_pred, prefix=""):
    """
    Compute R², RMSE, MAE in both log-space and original scale.
    Returns a dict of metric_name -> value.
    """
    m = {
        f"{prefix}log_r2": r2_score(y_true, y_pred),
        f"{prefix}log_rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        f"{prefix}orig_r2": r2_score(np.exp(y_true), np.exp(y_pred)),
        f"{prefix}orig_rmse": np.sqrt(mean_squared_error(np.exp(y_true), np.exp(y_pred))),
        f"{prefix}orig_mae": mean_absolute_error(np.exp(y_true), np.exp(y_pred)),
    }
    return m


def train_and_evaluate_models(X_train, y_train, X_test, y_test, models,
                               experiment_path=MLFLOW_EXPERIMENT):
    """
    Train each model, evaluate, and log to MLflow.
    `models` is a dict of {name: sklearn_estimator}.
    Returns dict of {name: {model, metrics...}}.
    """
    import warnings
    warnings.filterwarnings('ignore')
    mlflow.set_experiment(experiment_path)
    results = {}

    for name, model in models.items():
        with mlflow.start_run(run_name=name):
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            metrics = compute_metrics(y_test, y_pred)

            mlflow.log_params(model.get_params())
            mlflow.log_metrics(metrics)
            mlflow.set_tag("model_type", name)
            mlflow.set_tag("target_transform", "log")
            sig = mlflow.models.infer_signature(X_train, y_pred)
            mlflow.sklearn.log_model(model, artifact_path="model", signature=sig)

            results[name] = {"model": model, **metrics}
            print(f"{name:20s}  log_R²={metrics['log_r2']:.4f}  orig_R²={metrics['orig_r2']:.4f}")

    best = max(results, key=lambda k: results[k]['log_r2'])
    print(f"\n>>> Best: {best} (log R² = {results[best]['log_r2']:.4f})")
    return results, best


def grid_search_tune(X_train, y_train, param_grid, random_state=RANDOM_STATE):
    """
    Run GridSearchCV on XGBoost with the given param grid.
    Returns the fitted GridSearchCV object.
    """
    from xgboost import XGBRegressor

    n_combos = np.prod([len(v) for v in param_grid.values()])
    print(f"GridSearchCV: {n_combos} combos x 5 folds = {n_combos * 5} fits")

    gs = GridSearchCV(
        XGBRegressor(random_state=random_state, objective="reg:squarederror", n_jobs=-1),
        param_grid=param_grid,
        cv=5, scoring="r2", n_jobs=-1, verbose=1, refit=True,
    )
    gs.fit(X_train, y_train)
    print(f"Best CV R²: {gs.best_score_:.4f}")
    print(f"Best params: {gs.best_params_}")
    return gs


def evaluate_and_log_final(model, X_train, y_train, X_test, y_test,
                           model_tag, experiment_path=MLFLOW_EXPERIMENT,
                           best_cv_r2=None):
    """
    Evaluate the final tuned model in both log and original space,
    then log everything to MLflow.
    """
    mlflow.set_experiment(experiment_path)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    metrics = {
        **compute_metrics(y_train, y_train_pred, prefix="train_"),
        **compute_metrics(y_test, y_test_pred, prefix="test_"),
    }
    if best_cv_r2 is not None:
        metrics["cv_r2"] = best_cv_r2

    with mlflow.start_run(run_name=f"{model_tag}_tuned_final"):
        mlflow.log_params(model.get_params())
        mlflow.log_metrics(metrics)
        mlflow.set_tag("model_type", f"{model_tag}_tuned")
        mlflow.set_tag("target_transform", "log")
        mlflow.set_tag("tuning_method", "GridSearchCV")
        sig = mlflow.models.infer_signature(X_train, y_test_pred)
        mlflow.sklearn.log_model(model, artifact_path="model", signature=sig)

    print(f"Logged {model_tag} — Test log-R²={metrics['test_log_r2']:.4f}, "
          f"Test orig-R²={metrics['test_orig_r2']:.4f}")
    return metrics


def register_model(model_tag, model_name=MODEL_NAME_UC,
                   experiment_path=MLFLOW_EXPERIMENT):
    """
    Register the latest tuned model run to Unity Catalog.
    """
    mlflow.set_registry_uri("databricks-uc")
    run_id = mlflow.search_runs(
        experiment_names=[experiment_path],
        filter_string=f"tags.model_type = '{model_tag}_tuned'",
        order_by=["start_time DESC"],
        max_results=1,
    ).iloc[0].run_id
    model_uri = f"runs:/{run_id}/model"
    mv = mlflow.register_model(model_uri=model_uri, name=model_name)
    print(f"Registered {model_name} v{mv.version} (run {run_id})")
    return mv



Step 4 functions defined.


In [0]:

#  PREDICTION

def load_latest_model(model_name=MODEL_NAME_UC):
    """
    Load the latest version of the registered model from Unity Catalog.
    Returns (model, version_number).
    """
    client = mlflow.MlflowClient()
    latest_version = max(
        int(v.version) for v in client.search_model_versions(f"name='{model_name}'")
    )
    model_uri = f"models:/{model_name}/{latest_version}"
    model = mlflow.pyfunc.load_model(model_uri)
    print(f"Loaded {model_name} v{latest_version}")
    return model, latest_version


def predict_dielectric(model, prediction_table=PREDICTION_TABLE):
    """
    Run predictions on the prediction dataset.
    Returns a pandas DataFrame with id, formula, predicted_dielectric_constant
    (inverse log-transformed back to original scale).
    """
    pred_df = spark.table(prediction_table).toPandas()
    ids = pred_df['id']
    formulas = pred_df['formula']
    feature_cols = [c for c in pred_df.columns if c not in ID_COLS + [TARGET]]
    X_pred = pred_df[feature_cols]

    y_pred_log = model.predict(X_pred)
    y_pred = np.exp(y_pred_log)  # inverse log-transform

    results = pd.DataFrame({
        'id': ids,
        'formula': formulas,
        'predicted_dielectric_constant': y_pred,
    })
    print(f"Predictions generated for {len(results)} materials")
    return results


def run_prediction():
    """
    Full Step 5 pipeline:
    Load latest model → predict on unseen data → display results.
    """
    model, version = load_latest_model()
    results = predict_dielectric(model)
    display(spark.createDataFrame(results))
    return results



Step 5 functions defined.


In [0]:

# FULL PIPELINE RUNNER 


def run_full_pipeline(skip_ingestion=False, skip_merging=False, skip_training=False):
    """
    Run the entire dielectric constant ML pipeline end-to-end.

    Parameters
    ----------
    skip_ingestion : bool
        If True, skip Step 1 (assumes silver table already exists).
    skip_merging : bool
        If True, skip Step 2 (assumes featurized_merged already exists).
    skip_training : bool
        If True, skip Step 4 (uses existing registered model for prediction).
    """
    from xgboost import XGBRegressor

    # Step 1
    if not skip_ingestion:
        print("\n" + "="*60 + "\n STEP 1: Data Ingestion\n" + "="*60)
        run_data_ingestion()

    # Step 2
    if not skip_merging:
        print("\n" + "="*60 + "\n STEP 2: Feature Merging\n" + "="*60)
        run_feature_merging()

    # Step 3
    print("\n" + "="*60 + "\n STEP 3: Feature Selection & Split\n" + "="*60)
    train_df, test_df, pred_df, scaler, feature_cols = run_feature_selection_and_split()

    # Step 4
    if not skip_training:
        print("\n" + "="*60 + "\n STEP 4: Model Training\n" + "="*60)
        train_df, test_df = load_train_test_pandas()
        X_train, y_train, X_test, y_test = prepare_xy(train_df, test_df)

        models = {
            "XGBoost": XGBRegressor(
                random_state=42, n_estimators=1000, max_depth=5,
                learning_rate=0.05, min_child_weight=5, subsample=0.8,
                colsample_bytree=0.8, reg_alpha=1, reg_lambda=10,
                objective="reg:squarederror", n_jobs=-1,
            ),
        }
        results, best = train_and_evaluate_models(X_train, y_train, X_test, y_test, models)

        param_grid = {
            "n_estimators": [1000, 2000], "max_depth": [3, 5],
            "learning_rate": [0.01, 0.05], "subsample": [0.7, 0.8],
            "colsample_bytree": [0.7, 0.8], "reg_alpha": [1], "reg_lambda": [5],
        }
        gs = grid_search_tune(X_train, y_train, param_grid)
        evaluate_and_log_final(gs.best_estimator_, X_train, y_train, X_test, y_test,
                               best, best_cv_r2=gs.best_score_)
        register_model(best)

    # Step 5
    print("\n" + "="*60 + "\n STEP 5: Prediction\n" + "="*60)
    results = run_prediction()
    return results


Full pipeline runner defined.

✅ All utility functions ready. Import this notebook with %run or call individual step functions.
